# Coding attention mechanisms

Sequence processing one item at a time fails due to lack of context (think: translating a text string). Problem is commonly addressed by using encoder-decoder architectures. 

Problem with RNNs is that they process text sequentially, trying to capture the entire meaning of the input sequence from the final hidden state. Thus, a big limitation of RNNs is that they are unable to directly access earlier hidden states from the encoder during the decoding phase. Consequently, it relies solely on the current hidden state to encapsulate all relevant information. The RNN must remember the entire encoded input input in a single hidden state before passing it to the decoder. This can lead to loss of context, especially in sequences with dependencies spanning long distances.

Alternatively, the self-attention mechanism of the transformer architecture allows each position in the input sequence to consider the relevancy of all other positions in the same sequence. The self in self-attention refers to the computation of attention weights by relating different positions within a single input sequence. It assesses and learns the relationships and dependencies between various parts of the input itself. 

Self-attention calculates context vectors $z^{(i)}$ for each element $x^{(i)}$ of the input sequence. A context vector is an enriched embedding vector, incorporating information from all other other elements to create an enriched representation of each element in an input sequence. 

In [27]:
import torch
# Sample input sequence already embedded as 3-dimensional vectors 
inputs = torch.tensor(
    [[0.43, 0.15, 0.89], # Your    x^1 
     [0.55, 0.87, 0.66], # journey x^2
     [0.57, 0.85, 0.64], # starts  x^3
     [0.22, 0.58, 0.33], # with    x^4
     [0.77, 0.25, 0.10], # one     x^5
     [0.05, 0.80, 0.55]] # step    x^6
)

In [28]:
# First step to self-attention is to compute intermediate values
# called attention scores. These scores determine how much each token should
# attend to every other token in the sequence.  
query = inputs[1]
# Second token input serves as the query token
attn_scores_2 = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs):
    # Attention scores are computed as the dot product between the 
    # query and each token embedding is the dot product between the 
    # query and each token embedding
    attn_scores_2[i] = torch.dot(query, x_i)

# NOTE: The dot product can be viewed as a measure of similarity as it
# quantifies how closely two vectors are aligned: a high dot product indicates
# a greater degree of alignment. In context of self-attention, dot product
# determines extent to which each element in a sequence focuses on any other
# element: the higher the dot product, the higher the similarity and 
# attention score between two elements.
print(attn_scores_2)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


In [29]:
# Normalize so sum to 1
attn_weights = torch.softmax(attn_scores_2, dim=0)
# Softmax ensures positive output and permits interpretation as 
# probabilities/relative importance
print(attn_weights)
print(attn_weights.sum())

tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
tensor(1.)


In [30]:
# Context vector is computed by multiplying the embedded 
# input tokens x^(i) with the corresponding attention weights z^(2) and
# them summing the resulting vectors. Thus, the context vector, z^(2), is 
# the weighted sum of all input vectors, obtained by multiplying each input
# vector with corresponding attention weight.
# ---
# Second input token is the query
query = inputs[1]
context_vec_2 = torch.zeros(query.shape)
for i, x_i in enumerate(inputs):
    # Multiply embedding vector with corresponding attention weight
    context_vec_2 += x_i * attn_weights[i]
print(context_vec_2)

tensor([0.4419, 0.6515, 0.5683])


In [31]:
# Using matrix multiplication to calculate attention weights
attn_scores = inputs @ inputs.T 
attn_weights = torch.softmax(attn_scores, dim=-1)
print(attn_weights)

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])


In [32]:
# All context vectors via matrix multiplication
all_context_vecs = attn_weights @ inputs
print(all_context_vecs)

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


In [33]:
print(context_vec_2)
print(all_context_vecs[1])

tensor([0.4419, 0.6515, 0.5683])
tensor([0.4419, 0.6515, 0.5683])


## Self-attention with trainable weights 

The GPT models use a self-attention variant called scaled dot-product attention. The goal is still to copmute context vectors as weighted sums over input vectors specific to a certain input element. This approach, however, introduces trainable weight matrices $W_q$, $W_k$, $W_v$. Matrices project the input tokens, $x^{(i)}$, into query, key and value vectors. 

Intuition for queries, keys, and values in self‑attention is to think of it like a search engine inside the model. For each token in a sentence:

- Query (Q): What am I looking for?
- Key (K): What do I offer / what information do I represent?
- Value (V): What information should I pass along if selected?

Self-attention works by:

- Comparing queries with all keys → determines relevance (similarity scores)
- Using those scores to weight the values
- Producing a weighted sum → the output representation

**Example:** "The animal didn’t cross the street because it was too tired."

When processing the word/token for "it":

- Query asks: Who does "it" refer to?
- Keys: representations of "animal", "street", etc.
- Values: their contextual meanings

The model:

- Computes similarity between the query ("it") and keys
- Learns that "animal" is most relevant
- Pulls its value (meaning) into the representation of "it"

In [34]:
# Use second input element for illustration
x_2 = inputs[1]
# Input embedding size, d=3
d_in = inputs.shape[1]
# Output embedding size, d_out=2
d_out = 2

# Initiallize weight matrices
W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

In [35]:
# Compute query, key and value vectors
query_2 = x_2 @ W_query
key_2 = x_2 @ W_key
value_2 = x_2 @ W_value
print(query_2)

tensor([1.3730, 1.5279])


In [36]:
# Obtain keys and values via matrix multiplication
keys = inputs @ W_key 
values = inputs @ W_value 
print(keys.shape)
print(values.shape)

torch.Size([6, 2])
torch.Size([6, 2])


In [37]:
# Compute attention scores
keys_2 = keys[1]
attn_score_22 = query_2.dot(keys_2)
print(attn_score_22)

tensor(3.0859)


In [38]:
# All attention scores for a given query
attn_scores_2 = query_2 @ keys.T
print(attn_scores_2)

tensor([2.0002, 3.0859, 3.0539, 1.7121, 1.6169, 2.1338])


In [39]:
# Attention weights by scaling attention scores and using softmax
# Note that scale scores by dividing by square root of embedding dimension 
# of the keys
d_k = keys.shape[-1]
# Similar to computing context vector as weighted sum over input vectors,
# now compute context vector as weighted sum over value vectors
attn_weights_2 = torch.softmax(attn_scores_2 / d_k ** 0.5, dim=-1)
print(attn_weights_2)

tensor([0.1260, 0.2714, 0.2654, 0.1027, 0.0961, 0.1384])


In [40]:
# In matrix multiplication
context_vec_2 = attn_weights_2 @ values
print(context_vec_2)

tensor([1.4410, 1.0669])


## Self-attention class

In [41]:
import torch.nn as nn


class SelfAttention_v1(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        # Initialize trainable weight matrices
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key =   nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x):
        keys = x @ self.W_key 
        queries = x @ self.W_query
        values = x @ self.W_value
        attn_scores = queries @ keys.T 
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1] ** 0.2, dim=-1
        )
        context_vec = attn_weights @ values 
        return context_vec
    
torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in, d_out)
print(sa_v1(inputs))

tensor([[0.3038, 0.8154],
        [0.3115, 0.8336],
        [0.3112, 0.8328],
        [0.2980, 0.8018],
        [0.2956, 0.7961],
        [0.3031, 0.8139]], grad_fn=<MmBackward0>)


In [42]:
# Improving self-attention by utilizing improve weight initialization 
# in nn.Linear layers, and effectively perform matrix multiplication
# when bias units are disabled.


class SelfAttention_v2(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        # Initialize trainable weight matrices
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key =   nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        keys =    self.W_key(x)
        queries = self.W_query(x)
        values =  self.W_value(x)
        attn_scores = queries @ keys.T 
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1] ** 0.2, dim=-1
        )
        context_vec = attn_weights @ values 
        return context_vec
    
torch.manual_seed(123)
sa_v2 = SelfAttention_v2(d_in, d_out)
print(sa_v2(inputs))

tensor([[-0.5351, -0.1049],
        [-0.5334, -0.1084],
        [-0.5334, -0.1083],
        [-0.5301, -0.1079],
        [-0.5318, -0.1067],
        [-0.5304, -0.1085]], grad_fn=<MmBackward0>)


## Hiding future words with causal attention

For many LLM tasks, you will want the self-attention to consider only the tokens that appear prior to the current position when predicting the next token in a sequence. Casual/masked attention restricts the model to only consider current and previous inputs in a sequencewhen processing any given tokenand computing attention scores. 

In [43]:
# Reuses the query and key weight matrices of the SelAttention_v2 object
queries = sa_v2.W_query(inputs)
keys = sa_v2.W_key(inputs)
attn_scores = queries @ keys.T
attn_weights = torch.softmax(attn_scores / keys.shape[-1] ** 0.5, dim=-1)
print(attn_weights)

tensor([[0.1717, 0.1762, 0.1761, 0.1555, 0.1627, 0.1579],
        [0.1636, 0.1749, 0.1746, 0.1612, 0.1605, 0.1652],
        [0.1637, 0.1749, 0.1746, 0.1611, 0.1606, 0.1651],
        [0.1636, 0.1704, 0.1702, 0.1652, 0.1632, 0.1674],
        [0.1667, 0.1722, 0.1721, 0.1618, 0.1633, 0.1639],
        [0.1624, 0.1709, 0.1706, 0.1654, 0.1625, 0.1682]],
       grad_fn=<SoftmaxBackward0>)


In [44]:
# Use `tril` to create a mask where the values above the diagonal are 
# zero
context_length = attn_scores.shape[0]
mask_simple = torch.tril(torch.ones(context_length, context_length))
print(mask_simple)

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


In [45]:
# Multiply mask with attention weights to zero-out values above diagonal
masked_simple = mask_simple * attn_weights
print(masked_simple)

tensor([[0.1717, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1636, 0.1749, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1637, 0.1749, 0.1746, 0.0000, 0.0000, 0.0000],
        [0.1636, 0.1704, 0.1702, 0.1652, 0.0000, 0.0000],
        [0.1667, 0.1722, 0.1721, 0.1618, 0.1633, 0.0000],
        [0.1624, 0.1709, 0.1706, 0.1654, 0.1625, 0.1682]],
       grad_fn=<MulBackward0>)


In [46]:
# Re-nomarlize attention weights to unit sum
row_sums = masked_simple.sum(dim=-1, keepdim=True)
masked_simple_norm = masked_simple / row_sums
print(masked_simple_norm)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4833, 0.5167, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3190, 0.3408, 0.3402, 0.0000, 0.0000, 0.0000],
        [0.2445, 0.2545, 0.2542, 0.2468, 0.0000, 0.0000],
        [0.1994, 0.2060, 0.2058, 0.1935, 0.1953, 0.0000],
        [0.1624, 0.1709, 0.1706, 0.1654, 0.1625, 0.1682]],
       grad_fn=<DivBackward0>)


In [ ]:
# More efficient masking uses -inf in stead of zero above the diagonal.
# In softmax, exp(-inf) = 0. USing diagonal=1 offsets the diagonal one row
# up.
mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
masked = attn_scores.masked_fill(mask.bool(), -1.0 * torch.inf)
print(masked)

tensor([[0.3111,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.1655, 0.2602,   -inf,   -inf,   -inf,   -inf],
        [0.1667, 0.2602, 0.2577,   -inf,   -inf,   -inf],
        [0.0510, 0.1080, 0.1064, 0.0643,   -inf,   -inf],
        [0.1415, 0.1875, 0.1863, 0.0987, 0.1121,   -inf],
        [0.0476, 0.1192, 0.1171, 0.0731, 0.0477, 0.0966]],
       grad_fn=<MaskedFillBackward0>)


In [50]:
attn_weights = torch.softmax(masked / keys.shape[-1] ** 0.5, dim=1)
print(attn_weights)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4833, 0.5167, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3190, 0.3408, 0.3402, 0.0000, 0.0000, 0.0000],
        [0.2445, 0.2545, 0.2542, 0.2468, 0.0000, 0.0000],
        [0.1994, 0.2060, 0.2058, 0.1935, 0.1953, 0.0000],
        [0.1624, 0.1709, 0.1706, 0.1654, 0.1625, 0.1682]],
       grad_fn=<SoftmaxBackward0>)


## Masking attention weights with dropout

Dropout is a technique in deep learning where randomly selected hidden layer units are ignored during training. In models like GPT, dropout is typically applied at two specific times: after calculting the attention weights or after applying attention weights to the value vectors.

In [ ]:
torch.manual_seed(123)
# Dropout-rate of 50%
dropout = torch.nn.Dropout(0.5)
example = torch.ones(6, 6)
print(dropout(example))
# NOTE: To compensate for the reduction in active elements, values of 
# remaining elements are scaled up by a factor of 1 / 0.5 = 2. Scaling is 
# important to maintain influence of attention mechanism.

tensor([[2., 2., 0., 2., 2., 0.],
        [0., 0., 0., 2., 0., 2.],
        [2., 2., 2., 2., 0., 2.],
        [0., 2., 2., 0., 0., 2.],
        [0., 2., 0., 2., 0., 2.],
        [0., 2., 2., 2., 2., 0.]])


In [54]:
torch.manual_seed(123)
print(dropout(attn_weights))

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.6380, 0.6816, 0.6804, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.5090, 0.5085, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.4120, 0.0000, 0.3869, 0.0000, 0.0000],
        [0.0000, 0.3418, 0.3413, 0.3308, 0.3249, 0.0000]],
       grad_fn=<MulBackward0>)


## Causal attention class with dropout

In [ ]:
# Stack two samples to simulate batch data
batch = torch.stack((inputs, inputs), dim=0)
print(batch.shape)

torch.Size([2, 6, 3])


In [56]:
# Incorporates causal attention and dropout modifications


class CausalAttention(nn.Module):
    def __init__(
        self, d_in, d_out, context_length, dropout, qkv_bias=False
    ):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        # For auto handling of moving buffers to CPU/GPU.
        self.register_buffer(
            "mask", 
            torch.triu(
                torch.ones(context_length, context_length), 
                diagonal=1
            )
        )

    def forward(self, x):
        # batch, tokens and input embedding dimension
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        # Transpose to keep batch dimension first
        attn_scores = queries @ keys.transpose(1, 2)
        # In pytorch, ops with trailing underscore are performed in-place, 
        # avoiding unneceøssary memory copies
        attn_scores.masked_fill_(
            self.mask.bool()[:num_tokens, :num_tokens], -1.0 * torch.inf
        )
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1] ** 0.5, dim=-1
        )
        attn_weights = self.dropout(attn_weights)

        context_vec = attn_weights @ values
        return context_vec
    

torch.manual_seed(123)
context_length = batch.shape[1]
ca = CausalAttention(d_in, d_out, context_length, 0.0)
context_vecs = ca(batch)
print(context_vecs.shape)

torch.Size([2, 6, 2])


## Multi-head attention

Multi-head attention are built by stacking multiple causal attention modules. I.e., creating multiple instances of self-attention. 

In [ ]:
class MultiHeadAttentionWrapper(nn.Module):
    def __init__(
        self, 
        d_in, 
        d_out, 
        context_length, 
        dropout, 
        num_heads, 
        qkv_bias=False    
    ):
        super().__init__()
        self.heads = nn.ModuleList(
            [CausalAttention(
                d_in, d_out, context_length, dropout, qkv_bias
            )
            for _ in range(num_heads)]
        )

    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=-1)
    

torch.manual_seed(123)
# Number of tokens
context_length = batch.shape[1]
d_in, d_out = 3, 2
mha = MultiHeadAttentionWrapper(
    d_in, d_out, context_length, 0.0, num_heads=2
)
context_vecs = mha(batch)
print(context_vecs)
print(context_vecs.shape)

tensor([[[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]],

        [[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]]], grad_fn=<CatBackward0>)
torch.Size([2, 6, 4])


In [65]:
# Improving MultiHeadAttentionWrapper by not processing multiple single-
# head attention modules sequentially, but in parallel. Achieve this by 
# computing outputs via matrix multiplication.
class MultiHeadAttention(nn.Module):
    def __init__(
        self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False
    ):
        super().__init__()
        assert (d_out % num_heads == 0) # sanity check

        self.d_out = d_out
        self.num_heads = num_heads
        # Reduces projection dim to match desired output dim
        self.head_dim = d_out // num_heads
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        # Combine heads outputs in a linear layer
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(
                torch.ones(context_length, context_length),
                diagonal=1
            )
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        # Implicitly split matrix by adding a num_heads dimention. Then 
        # unroll the last dim: 
        # (b, tokens, d_out) -> (b, tokens, n_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)
        # Transpose from (b, tokens, n_heads, heads_dim) to 
        # (b, n_heads, tokens, heads_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # Computes dot product for each head
        attn_scores = queries @ keys.transpose(2, 3)
        # Masks are truncated to number of tokens
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -1.0 * torch.inf)

        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1] ** 0.5, dim=-1
        )
        attn_weights = self.dropout(attn_weights)
        # Tensor shape: (b, tokens, n_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2)
        # Combines heads, where d_out = n_heads * head_dim
        context_vec = context_vec.contiguous().view(
            b, num_tokens, self.d_out
        )
        # Adds extra linear projection
        context_vec = self.out_proj(context_vec)
        return context_vec
    

torch.manual_seed(123)
batch_size, context_length, d_in = batch.shape
d_out = 2
mha = MultiHeadAttention(
    d_in, d_out, context_length, 0.0, num_heads=2
)
context_vecs = mha(batch)
print(context_vecs)
print(context_vecs.shape)

tensor([[[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]],

        [[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]]], grad_fn=<ViewBackward0>)
torch.Size([2, 6, 2])
